In [1]:
import os
from collections import Counter

path = "E:/DoAn2026/Stickers_Data"
file_extensions = []

for root, dirs, files in os.walk(path):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        file_extensions.append(ext)

print("Thống kê các loại file bạn đang có:")
print(Counter(file_extensions))

Thống kê các loại file bạn đang có:
Counter({'.lock': 2232, '.metadata': 2232, '.jpg': 1447, '.png': 455, '.jpeg': 121, '.gif': 76, '.json': 73, '': 34, '.parquet': 26, '.csv': 9, '.zip': 8, '.jsonl': 2, '.ipynb': 1, '.xlsx': 1, '.txt': 1, '.z01': 1, '.z02': 1, '.z03': 1, '.z04': 1, '.z05': 1, '.z06': 1, '.001': 1, '.002': 1, '.003': 1, '.004': 1, '.005': 1, '.006': 1})


In [6]:
import os
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch

# 1. Kết nối DB
conn = psycopg2.connect(
    dbname="KanvaPro",        # Tên DB bạn đã tạo (KanvaPro thay vì your_db)
    user="postgres", 
    password="Superhero3@",  # Mật khẩu bạn dùng trong config Node.js
    host="localhost",
    port="5432"
)
cur = conn.cursor()

# 2. Lấy UUID của Category "Stickers"
cur.execute("SELECT id FROM asset_categories WHERE slug = 'stickers' LIMIT 1")
res = cur.fetchone()
category_id = res[0] if res else None

source_dir = "E:/DoAn2026/Stickers_Data"

def import_parquet(file_path):
    print(f"--- Đang xử lý file: {os.path.basename(file_path)} ---")
    df = pd.read_parquet(file_path)
    
    # Chuẩn hóa tên cột (Vì mỗi dataset có thể đặt tên cột khác nhau)
    # Giả sử chúng ta cần: name, url, tags
    # Bạn hãy kiểm tra df.columns để mapping cho đúng
    
    data_to_insert = []
    for _, row in df.iterrows():
        # Map dữ liệu từ Parquet sang cấu trúc bảng assets của bạn
        data_to_insert.append((
            row.get('TEXT', 'Unnamed Sticker'), # name
            'sticker',                          # asset_type enum
            row.get('URL', ''),                # url
            row.get('URL', ''),                # thumbnail_url
            0,                                 # file_size (nếu không có)
            row.get('WIDTH', 512),             # width
            row.get('HEIGHT', 512),            # height
            False,                             # is_premium
            category_id,                       # category_id
            ['sticker', 'laion'],              # tags (array)
            '{"source": "LAION-5B"}'           # metadata (jsonb)
        ))

    # Insert theo batch (mỗi lần 1000 dòng để nhanh hơn)
    query = """
        INSERT INTO assets (name, type, url, thumbnail_url, file_size, width, height, is_premium, category_id, tags, metadata)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """
    execute_batch(cur, query, data_to_insert, page_size=1000)
    conn.commit()
    print(f"Đã xong file {os.path.basename(file_path)}")

# Quét tất cả file .parquet trong các thư mục con
for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.endswith(".parquet"):
            import_parquet(os.path.join(root, file))

cur.close()
conn.close()
print("--- HOÀN TẤT TOÀN BỘ ---")

--- Đang xử lý file: train-00000-of-00001.parquet ---
Đã xong file train-00000-of-00001.parquet
--- Đang xử lý file: train-00000-of-00006.parquet ---
Đã xong file train-00000-of-00006.parquet
--- Đang xử lý file: train-00001-of-00006.parquet ---
Đã xong file train-00001-of-00006.parquet
--- Đang xử lý file: train-00002-of-00006.parquet ---
Đã xong file train-00002-of-00006.parquet
--- Đang xử lý file: train-00003-of-00006.parquet ---
Đã xong file train-00003-of-00006.parquet
--- Đang xử lý file: train-00004-of-00006.parquet ---
Đã xong file train-00004-of-00006.parquet
--- Đang xử lý file: train-00005-of-00006.parquet ---
Đã xong file train-00005-of-00006.parquet
--- Đang xử lý file: train-00000-of-00001.parquet ---
Đã xong file train-00000-of-00001.parquet
--- Đang xử lý file: train-00000-of-00001.parquet ---
Đã xong file train-00000-of-00001.parquet
--- Đang xử lý file: train-00000-of-00001.parquet ---
Đã xong file train-00000-of-00001.parquet
--- Đang xử lý file: train-00000-of-0000

In [ ]:
# %pip install pandas pyarrow psycopg2-binary

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 18.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import os
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch

# 1. Kết nối DB
conn = psycopg2.connect(
    dbname="KanvaPro", user="postgres", password="Superhero3@", host="localhost"
)
cur = conn.cursor()

# 2. Đảm bảo có Category 'stickers' (Sửa lỗi NoneType tại đây)
cur.execute("""
    INSERT INTO asset_categories (name, slug) 
    VALUES ('Stickers', 'stickers') 
    ON CONFLICT (slug) DO UPDATE SET name = EXCLUDED.name 
    RETURNING id
""")
category_id = cur.fetchone()[0]
conn.commit()

# 3. Đường dẫn lưu file
storage_dir = "E:/DoAn2026/KanvaPro/public/assets/stickers"
os.makedirs(storage_dir, exist_ok=True)

source_dir = "E:/DoAn2026/Stickers_Data"

def process_binary_parquet(file_path):
    print(f"--- Đang trích xuất: {os.path.basename(file_path)} ---")
    df = pd.read_parquet(file_path)
    
    data_to_insert = []
    
    for index, row in df.iterrows():
        try:
            # Lấy dữ liệu bytes (Hugging Face thường để trong dict {'bytes': ...})
            img_data = row['image']['bytes']
            desc = row.get('description', 'Pusheen Sticker')
            
            # Tạo tên file: Tên folder + tên file + index để tránh trùng
            parent_folder = os.path.basename(os.path.dirname(os.path.dirname(file_path)))
            filename = f"{parent_folder}_{index}.png"
            local_path = os.path.join(storage_dir, filename)
            
            # Ghi file ảnh
            with open(local_path, 'wb') as f:
                f.write(img_data)
            
            # Đường dẫn mà Frontend sẽ gọi
            internal_url = f"/assets/stickers/{filename}"
            
            data_to_insert.append((
                desc[:200], 'sticker', internal_url, internal_url,
                len(img_data), 512, 512, False, category_id,
                ['sticker', parent_folder, 'pusheen'],
                '{"source": "binary_parquet"}'
            ))
            
        except Exception as e:
            continue

    if data_to_insert:
        query = """
            INSERT INTO assets (name, type, url, thumbnail_url, file_size, width, height, is_premium, category_id, tags, metadata)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """
        execute_batch(cur, query, data_to_insert, page_size=500)
        conn.commit()
        print(f"Đã lưu {len(data_to_insert)} sticker từ file này.")

# Quét toàn bộ thư mục
for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.endswith(".parquet"):
            process_binary_parquet(os.path.join(root, file))

cur.close()
conn.close()
print("--- TẤT CẢ ĐÃ XONG! ---")

--- Đang trích xuất: train-00000-of-00001.parquet ---
Đã lưu 176 sticker từ file này.
--- Đang trích xuất: train-00000-of-00006.parquet ---
--- Đang trích xuất: train-00001-of-00006.parquet ---
--- Đang trích xuất: train-00002-of-00006.parquet ---
--- Đang trích xuất: train-00003-of-00006.parquet ---
--- Đang trích xuất: train-00004-of-00006.parquet ---
--- Đang trích xuất: train-00005-of-00006.parquet ---
--- Đang trích xuất: train-00000-of-00001.parquet ---
Đã lưu 122 sticker từ file này.
--- Đang trích xuất: train-00000-of-00001.parquet ---
Đã lưu 46 sticker từ file này.
--- Đang trích xuất: train-00000-of-00001.parquet ---
Đã lưu 75 sticker từ file này.
--- Đang trích xuất: train-00000-of-00001.parquet ---
Đã lưu 118 sticker từ file này.
--- Đang trích xuất: train-00000-of-00001.parquet ---
Đã lưu 148 sticker từ file này.
--- Đang trích xuất: train-00000-of-00001.parquet ---
Đã lưu 113 sticker từ file này.
--- Đang trích xuất: train-00000-of-00001.parquet ---
Đã lưu 40 sticker từ f

In [12]:
import pandas as pd
import os

source_dir = "E:/DoAn2026/Stickers_Data"

# Tự động tìm file parquet đầu tiên để kiểm tra cấu trúc
found_file = None
for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.endswith(".parquet"):
            found_file = os.path.join(root, file)
            break
    if found_file: break

if found_file:
    print(f"--- Đang kiểm tra file: {found_file} ---")
    df = pd.read_parquet(found_file)
    print("\n[DANH SÁCH CỘT THỰC TẾ]:")
    print(df.columns.tolist())
    print("\n[DỮ LIỆU MẪU]:")
    print(df.head(1))
else:
    print("Không tìm thấy bất kỳ file .parquet nào trong thư mục E:/DoAn2026/Stickers_Data")

--- Đang kiểm tra file: E:/DoAn2026/Stickers_Data\alexanz_pusheen_stickers\data\train-00000-of-00001.parquet ---

[DANH SÁCH CỘT THỰC TẾ]:
['image', 'description']

[DỮ LIỆU MẪU]:
                                               image  \
0  {'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...   

                                         description  
0  Sticker of Pusheen. A cartoon image of a gray ...  
